<a href="https://colab.research.google.com/github/cuiandrew08-lab/LiDARFusionLearning/blob/main/LiDARtrain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import numpy as np

from google.colab import drive
drive.mount("/content/drive")

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
from tqdm.notebook import tqdm

import tensorflow as tf

TORCH_version = torch.__version__.split('+')[0]
#CUDA_version = torch.version.cuda.replace('.', '')

#!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-{TORCH_version}+cu{CUDA_version}.html

#import torch_sparse

from scipy.ndimage import maximum_filter

import sys

#!pip install import-ipynb

!pip install open3d plotly

import open3d as o3d
import matplotlib.pyplot as plt

#import import_ipynb

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!npx degit google-research-datasets/Objectron/objectron objectron

sys.path.insert(0, '/content')

from objectron.dataset.iou import IoU as IoU3d
from objectron.dataset.box import Box

⠙⠹⠸⠼Need to install the following packages:
degit@3.6.1
Ok to proceed? (y) y

⠙⠹⠸⠼⠴⠦> cloned google-research-datasets/Objectron#HEAD to objectron
⠙

In [3]:
!pip install nuscenes-devkit &> /dev/null  # Install nuScenes.

from nuscenes.nuscenes import NuScenes
from nuscenes.utils.data_classes import LidarPointCloud
nusc_root = "/content/drive/MyDrive/LiDARFusion/nuscenes/datanuscenes"

nusc = NuScenes(version='v1.0-mini', dataroot=nusc_root, verbose=True)

Loading NuScenes tables for version v1.0-mini...
23 category,
8 attribute,
4 visibility,
911 instance,
12 sensor,
120 calibrated_sensor,
31206 ego_pose,
8 log,
10 scene,
404 sample,
31206 sample_data,
18538 sample_annotation,
4 map,
Done loading in 13.526 seconds.
Reverse indexing ...
Done reverse indexing in 0.1 seconds.


In [5]:
sys.path.insert(0, '/content/drive/MyDrive/LiDARFusion')
%cd /content/drive/MyDrive/LiDARFusion

from voxel_pointnet2 import PointNetSetAbstraction

from centerpoint import CenterPoint

from lidar_baseline import PillarsEncoder

#import CenterPointHead

/content/drive/MyDrive/LiDARFusion


features(Δt, intensity) are kept in separate matrices, and pasted together for encoding

In [21]:
test_scene = nusc.scene[0]

token_0 = test_scene["first_sample_token"]

cloud_sample = nusc.get("sample", token_0)
cloud_data = nusc.get("sample_data", cloud_sample["data"]["LIDAR_TOP"])

cloud_root = os.path.join(nusc_root, cloud_data["filename"])

cloud = LidarPointCloud.from_file(cloud_root)

cloud_feats = cloud.points[3:,:].T

In [23]:
def extract_feats(scene, data_root):

  scene_tokens = []
  token_0 = scene["first_sample_token"]

  while token_0 != "":
    scene_tokens.append(token_0)

    token_sample = nusc.get("sample", token_0)
    token_0 = token_sample["next"]

  scene_feats = []

  for i in range(len(scene_tokens)):
    cloud_sample = nusc.get("sample", scene_tokens[i])
    cloud_data = nusc.get("sample_data", cloud_sample["data"]["LIDAR_TOP"])

    cloud_root = os.path.join(data_root, cloud_data["filename"])

    cloud = LidarPointCloud.from_file(cloud_root)

    cloud_feats = cloud.points[3:, :].T

    scene_feats.append(cloud_feats)

  return scene_feats

In [37]:
lidar_token = cloud_sample['data']["LIDAR_TOP"]
root, boxes, _ = nusc.get_sample_data(lidar_token)